# AlpaJudge RB and RF inference on PhysicalAI-AV

This notebook downloads both checkpoint variants from `nvidia/alpajudge`, streams the frozen six-frame input from the NVIDIA [PhysicalAI-AV Dataset](https://huggingface.co/datasets/nvidia/PhysicalAI-Autonomous-Vehicles), and runs one reference-based and one reference-free score.

## Before you start

From the repository root, run `uv sync --locked --dev`, authenticate with `hf auth login`, and start this notebook with `uv run jupyter lab examples/alpajudge_pai.ipynb`. The Hugging Face account must be able to access both the model and dataset repositories. A CUDA GPU is required for the RF forward pass.

The rationale and meta-action strings below are manually authored example annotations for the fixed scene. PhysicalAI-AV supplies the camera frames, not these semantic labels. Replace all text fields together when evaluating another moment. This notebook is a smoke test, not a benchmark.

In [ ]:
import gc
import json
import os
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import physical_ai_av
import torch
from huggingface_hub import snapshot_download
from IPython.display import display

from alpajudge.reference_based import load_reference_based, score_reference_based
from alpajudge.reference_free import load_reference_free, score_reference_free

MODEL_ID = "nvidia/alpajudge"
MODEL_REVISION = os.environ.get("ALPAJUDGE_MODEL_REVISION", "main")
PAI_REVISION = os.environ.get("ALPAJUDGE_PAI_REVISION", "b719eea7f0a63619ef51ec7f54178af0937ef050")
CLIP_ID = os.environ.get("ALPAJUDGE_PAI_CLIP_ID", "030c760c-ae38-49aa-9ad8-f5650a545d26")
T0_US = int(os.environ.get("ALPAJUDGE_PAI_T0_US", "5100000"))
MODEL_CACHE = os.environ.get("ALPAJUDGE_MODEL_CACHE")
PAI_CACHE = os.environ.get("ALPAJUDGE_PAI_CACHE")
DEVICE = torch.device("cuda")

if not torch.cuda.is_available():
    raise RuntimeError("AlpaJudge-RF inference requires a CUDA GPU.")

print("device:", torch.cuda.get_device_name(DEVICE))
print("model revision:", MODEL_REVISION)
print("PAI revision:", PAI_REVISION)
print("sample:", CLIP_ID, T0_US)

## Download both checkpoints

The snapshot filter downloads only the two model subfolders. Set `ALPAJUDGE_MODEL_CACHE` when the default Hugging Face cache is too small. The default model revision may change as the hosted checkpoint is updated.

In [ ]:
checkpoint_root = Path(
    snapshot_download(
        repo_id=MODEL_ID,
        revision=MODEL_REVISION,
        allow_patterns=["reference-based/*", "reference-free/*"],
        cache_dir=MODEL_CACHE,
    )
)
for subfolder in ("reference-based", "reference-free"):
    config_path = checkpoint_root / subfolder / "config.json"
    if not config_path.is_file():
        raise FileNotFoundError(f"Missing checkpoint config: {config_path}")
print("checkpoint root:", checkpoint_root)

## Load the PhysicalAI-AV frames

AlpaJudge-RF uses front-wide followed by front-tele frames at `[-1.0, -0.5, 0.0]` seconds relative to the judged moment. PhysicalAI-AV returns the nearest decoded frame for each requested timestamp.

In [ ]:
def load_pai_frames(
    avdi, clip_id: str, t0_us: int
) -> tuple[torch.Tensor, torch.Tensor, dict[str, list[int]]]:
    """Load the frozen AlpaJudge-RF two-camera, three-frame input."""
    target_timestamps = np.asarray([t0_us - 1_000_000, t0_us - 500_000, t0_us], dtype=np.int64)
    camera_features = {
        "front_wide": avdi.features.CAMERA.CAMERA_FRONT_WIDE_120FOV,
        "front_tele": avdi.features.CAMERA.CAMERA_FRONT_TELE_30FOV,
    }
    frame_tensors = {}
    decoded_timestamps = {}
    for name, feature in camera_features.items():
        camera = avdi.get_clip_feature(clip_id, feature, maybe_stream=True)
        frames, timestamps = camera.decode_images_from_timestamps(target_timestamps)
        frame_tensor = torch.from_numpy(np.asarray(frames)).permute(0, 3, 1, 2).contiguous()
        if frame_tensor.ndim != 4 or frame_tensor.shape[:2] != (3, 3):
            raise ValueError(
                f"{name} decoded shape {tuple(frame_tensor.shape)}, expected [3,3,H,W]"
            )
        frame_tensors[name] = frame_tensor
        decoded_timestamps[name] = np.asarray(timestamps, dtype=np.int64).tolist()
    return (
        frame_tensors["front_wide"],
        frame_tensors["front_tele"],
        decoded_timestamps,
    )

In [ ]:
avdi = physical_ai_av.PhysicalAIAVDatasetInterface(revision=PAI_REVISION, cache_dir=PAI_CACHE)
front_wide_frames, front_tele_frames, decoded_timestamps = load_pai_frames(avdi, CLIP_ID, T0_US)
print("decoded timestamps:", decoded_timestamps)
print("wide shape:", tuple(front_wide_frames.shape))
print("tele shape:", tuple(front_tele_frames.shape))

fig, axes = plt.subplots(2, 3, figsize=(15, 6))
for row, (camera_name, frames) in enumerate(
    (("front-wide", front_wide_frames), ("front-tele", front_tele_frames))
):
    for column, frame in enumerate(frames):
        axes[row, column].imshow(frame.permute(1, 2, 0).cpu().numpy())
        offset = (-1.0, -0.5, 0.0)[column]
        axes[row, column].set_title(f"{camera_name} t={offset:+.1f}s")
        axes[row, column].axis("off")
fig.tight_layout()
display(fig)
plt.close(fig)

## Example text inputs

The fixed scene follows a white sedan through a construction-narrowed lane. Its egomotion shows gentle deceleration across both scoring windows. The candidate is intentionally shorter than the reference so RB measures alignment while RF checks it against the visual/action evidence.

In [ ]:
QUESTION = "What is the reasoning process of driving?"
REFERENCE_RATIONALE = (
    "The ego vehicle should gently decelerate and continue straight behind the white "
    "sedan, keeping a safe distance because construction equipment and cones narrow "
    "the lane."
)
CANDIDATE_RATIONALE = (
    "Continue straight and slow slightly behind the white sedan because the work zone "
    "narrows the usable lane."
)
META_ACTION_HISTORY = (
    "  Longitudinal: [-1.6,0.0) Gentle Deceleration\n"
    "  Lateral: [-1.6,0.0) Go Straight\n"
    "  Lane: [-1.6,0.0) Lane Keep"
)
META_ACTION_FUTURE = (
    "  Longitudinal: [0.0,6.4) Gentle Deceleration\n"
    "  Lateral: [0.0,6.4) Go Straight\n"
    "  Lane: [0.0,6.4) Lane Keep"
)

## Run AlpaJudge-RB

RB is text-only. It compares the candidate with the supplied reference and returns one alignment probability. With the recorded checkpoint and text inputs, the observed AlpaJudge-RB alignment was `0.999`. Displayed score values are truncated to three decimals.

In [ ]:
rb_model, rb_tokenizer = load_reference_based(
    str(checkpoint_root), subfolder="reference-based", device=DEVICE
)
rb_scores = score_reference_based(
    rb_model,
    rb_tokenizer,
    [
        {
            "question": QUESTION,
            "answer": REFERENCE_RATIONALE,
            "student": CANDIDATE_RATIONALE,
        }
    ],
)
if not torch.isfinite(rb_scores).all() or not torch.all((0 <= rb_scores) & (rb_scores <= 1)):
    raise ValueError(f"Invalid RB scores: {rb_scores}")
print("AlpaJudge-RB alignment:", rb_scores[0].item())

del rb_model
gc.collect()
torch.cuda.empty_cache()

## Run AlpaJudge-RF

RF reads the six PAI frames, the two example meta-action windows, and the candidate rationale. It returns five rubric probabilities and their aggregations. With the recorded checkpoint and fixed PAI frames, the observed output was:

```json
{
  "subscores": {
    "action_matches_gt": 0.791,
    "action_safe": 0.987,
    "components_grounded": 0.994,
    "components_complete": 0.927,
    "component_causal": 0.852
  },
  "action": 0.787,
  "component": 0.924,
  "overall": 0.842,
  "hard_action": 1.000,
  "hard_component": 1.000,
  "hard_overall": 1.000
}
```

Displayed score values are truncated to three decimals.

In [ ]:
rf_model, rf_processor = load_reference_free(
    str(checkpoint_root), subfolder="reference-free", device=DEVICE
)
with torch.autocast("cuda", dtype=torch.bfloat16):
    rf_scores = score_reference_free(
        rf_model,
        rf_processor,
        front_wide_frames=front_wide_frames,
        front_tele_frames=front_tele_frames,
        meta_action_history=META_ACTION_HISTORY,
        meta_action_future=META_ACTION_FUTURE,
        rationale=CANDIDATE_RATIONALE,
    )
score_values = list(rf_scores.subscores.values()) + [
    rf_scores.action,
    rf_scores.component,
    rf_scores.overall,
]
if not all(np.isfinite(value) and 0 <= value <= 1 for value in score_values):
    raise ValueError(f"Invalid RF scores: {rf_scores}")
print(json.dumps(asdict(rf_scores), indent=2))

## Interpret the smoke test

Both checks above validate that model download, registration, preprocessing, and GPU forward execution work. Scores are model outputs for one hand-authored example; they are not accuracy measurements or recommended thresholds.

The recorded smoke test used the notebook's fixed PAI sample at timestamp `5100000` µs. It ran on one NVIDIA H100 80GB HBM3 GPU with Python 3.12.3, PyTorch `2.8.0+cu128`, CUDA `12.8`, Transformers `4.57.6`, and `physical-ai-av` `0.2.2`. The RB model loaded in FP32; the RF model loaded in BF16 with SDPA attention and the notebook's BF16 CUDA autocast.

In that environment, two independent notebook kernels and two independent driver processes with three calls each produced eight observations per score. All 12 returned floating-point score values matched exactly across those observations with maximum absolute delta `0.0`; decoded frame hashes and processor input hashes also matched. Treat that as reproducibility evidence for the pinned inputs, checkpoint, and environment, not a cross-hardware bitwise guarantee.